In [2]:
import sys
import csv
import time
import numpy as np
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from my_rag import search as s
from my_rag.config import FAQ_FILE

TEST_FILE = ROOT / "evaluation" / "realistic_test_egyptian.csv"

positives = []
negatives = []
with open(TEST_FILE, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if "RAG" in row["route"].split("+") and row["expected_doc"].strip():
            positives.append({"question": row["question"], "docs": row["expected_doc"].split(";")})
        elif row["route"] == "REFUSE":
            negatives.append(row["question"])

with open(FAQ_FILE, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["source_doc"] == "out_of_scope_policy":
            negatives.append(row["question"])

print(len(positives), "with answer |", len(negatives), "outside the hotel")

c:\projects\HotelAI-Intelligent-Agent_Final_Data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1031 21:34:25.050000 15564 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 43447.59it/s]


105 with answer | 67 outside the hotel


In [4]:
at5 = 0
at20 = 0
times = []
pos_scores = []
misses = []

for item in positives:
    start = time.time()
    results = s.search(item["question"])
    times.append(time.time() - start)

    best, dense = s.hybrid_rank(item["question"])
    top5 = []
    for r in results:
        top5.append(r["source"])
    top20 = []
    for i in best:
        top20.append(s._chunks[i]["source"])

    if any(d in top5 for d in item["docs"]):
        at5 += 1
    else:
        misses.append((item["question"], item["docs"], top5[:3]))
    if any(d in top20 for d in item["docs"]):
        at20 += 1
    if results:
        pos_scores.append(results[0]["score"])

n = len(positives)
print("Recall@5 :", round(100 * at5 / n), "%")
print("Recall@20:", round(100 * at20 / n), "%")
print("time: avg", round(sum(times) / n, 3), "s | max", round(max(times), 3), "s")
print()
for q, docs, got in misses:
    print(q, "| expected:", docs, "| got:", got)

Recall@5 : 88 %
Recall@20: 97 %
time: avg 0.017 s | max 0.058 s

حاجز من بوكينج، أعدل منكم ولا منهم؟ | expected: ['booking_modification.txt'] | got: ['room_policies.txt', 'cancellation_process.txt', 'hotel_facilities.txt']
لو مشيت بدري يوم هتحسبوه عليا؟ | expected: ['booking_modification.txt', 'cancellation_policy.txt'] | got: ['check_in_check_out.txt', 'check_in_check_out.txt', 'no_show_policy.txt']
أوصل للجولد إزاي؟ | expected: ['loyalty_programs.txt'] | got: []
عايز دبل لتلاتة، أنا ومراتي وابننا 5 سنين | expected: ['children_policy.txt', 'room_types.txt'] | got: ['room_policies.txt', 'hotel_information.txt', 'room_amenities.txt']
عايز أحجز بس مش عارف أنهي أوضة تنفعني، إحنا اتنين وطفل | expected: ['room_types.txt', 'children_policy.txt'] | got: ['booking_policy.txt', 'booking_process.txt', 'booking_policy.txt']
بكام السويت تلات ليالي؟ | expected: ['pricing_and_rates.txt', 'room_types.txt'] | got: ['check_in_check_out.txt', 'noise_policy.txt', 'room_service.txt']
الكارت اترفض في حجز B

In [3]:
neg_scores = []
for q in negatives:
    results = s.search(q)
    if results:
        neg_scores.append(results[0]["score"])

print("with answer  : min", round(min(pos_scores), 3), "| median", round(float(np.median(pos_scores)), 3))
print("outside hotel: max", round(max(neg_scores), 3), "| median", round(float(np.median(neg_scores)), 3))
print()
for t in [0.40, 0.45, 0.50, 0.55, 0.60]:
    kept = 0
    for x in pos_scores:
        if x >= t:
            kept += 1
    blocked = 0
    for x in neg_scores:
        if x < t:
            blocked += 1
    print(t, "| keep answers:", round(100 * kept / len(pos_scores)), "% | block outside:", round(100 * blocked / len(neg_scores)), "%")

with answer  : min 0.296 | median 0.61
outside hotel: max 0.69 | median 0.54

0.4 | keep answers: 99 % | block outside: 3 %
0.45 | keep answers: 95 % | block outside: 7 %
0.5 | keep answers: 90 % | block outside: 22 %
0.55 | keep answers: 74 % | block outside: 58 %
0.6 | keep answers: 57 % | block outside: 75 %


In [4]:
anchors = []
faq_out = []
with open(FAQ_FILE, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["source_doc"] == "out_of_scope_policy":
            if row["split"] == "train":
                anchors.append(row["question"])
            else:
                faq_out.append(row["question"])

refuse = []
with open(TEST_FILE, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["route"] == "REFUSE":
            refuse.append(row["question"])

anchor_vecs = s._model.encode(anchors, normalize_embeddings=True).astype(np.float32)

def gap(question):
    q = s._model.encode([question], normalize_embeddings=True)[0].astype(np.float32)
    chunk_best = float(np.max(s._vectors @ q))
    anchor_best = float(np.max(anchor_vecs @ q))
    return anchor_best - chunk_best

pos_gaps = []
for item in positives:
    pos_gaps.append(gap(item["question"]))
refuse_gaps = []
for q in refuse:
    refuse_gaps.append(gap(q))
faq_gaps = []
for q in faq_out:
    faq_gaps.append(gap(q))

print(len(anchors), "anchors |", len(refuse), "refuse |", len(faq_out), "faq outside")
for margin in [-0.05, 0.0, 0.05, 0.1]:
    keep = 0
    for g in pos_gaps:
        if g <= margin:
            keep += 1
    block_r = 0
    for g in refuse_gaps:
        if g > margin:
            block_r += 1
    block_f = 0
    for g in faq_gaps:
        if g > margin:
            block_f += 1
    print(margin, "| keep answers:", round(100 * keep / len(pos_gaps)), "% | block refuse:", block_r, "/", len(refuse), "| block faq outside:", block_f, "/", len(faq_out))

40 anchors | 10 refuse | 17 faq outside
-0.05 | keep answers: 68 % | block refuse: 9 / 10 | block faq outside: 17 / 17
0.0 | keep answers: 88 % | block refuse: 8 / 10 | block faq outside: 17 / 17
0.05 | keep answers: 97 % | block refuse: 5 / 10 | block faq outside: 17 / 17
0.1 | keep answers: 100 % | block refuse: 3 / 10 | block faq outside: 17 / 17


In [5]:
MARGIN = 0.05

print("=== answers blocked by mistake:")
for item, g in zip(positives, pos_gaps):
    if g > MARGIN:
        print("   ", round(g, 3), item["question"])

print("=== outside questions that passed:")
for q, g in zip(refuse, refuse_gaps):
    if g <= MARGIN:
        print("   ", round(g, 3), q)

=== answers blocked by mistake:
    0.091 أوصل للجولد إزاي؟
    0.055 عايز أحجز وأستخدم نقطي كخصم
    0.057 ينفع أشرب شيشة في الأوضة؟
=== outside questions that passed:
    0.033 ممكن تساعدني في واجب الرياضيات؟
    -0.09 الدولار النهارده بكام؟
    -0.002 اكتبلي بوست لفيسبوك عن الأجازة
    0.007 تعرف تحجزلي في مطعم برا الفندق؟
    0.049 إيه رأيك في الانتخابات؟


In [6]:
extra_anchors = [
    "سعر اليورو النهارده كام؟",
    "البورصة عاملة إيه النهارده؟",
    "اشرحلي درس الكيميا ده",
    "حللي المسألة دي في الفيزيا",
    "اكتبلي إيميل لمديري إني تعبان",
    "اكتبلي كابشن لصورة على انستجرام",
    "احجزلي ترابيزة في مطعم في الزمالك",
    "فين أحسن مكان آكل فيه سمك في القاهرة؟",
    "مين كسب ماتش الأهلي امبارح؟",
    "إيه رأيك في الحكومة؟",
    "إيه آخر الأخبار النهارده؟",
    "احجزلي قطر لإسكندرية",
    "اعملي برنامج بايثون",
    "ازاي أعمل محشي؟",
    "ترشحلي فيلم أتفرج عليه؟",
    "فندق الهيلتون أسعاره كام؟",
    "الطيارة لدبي بكام؟",
    "Write me a poem about the sea",
    "Who won the football match yesterday?",
    "Recommend a good restaurant outside the hotel",
    "Help me with my programming homework",
    "What do you think about politics?",
]

anchor_vecs = s._model.encode(anchors + extra_anchors, normalize_embeddings=True).astype(np.float32)

pos_gaps = []
for item in positives:
    pos_gaps.append(gap(item["question"]))
refuse_gaps = []
for q in refuse:
    refuse_gaps.append(gap(q))

for margin in [0.0, 0.05, 0.1]:
    keep = 0
    for g in pos_gaps:
        if g <= margin:
            keep += 1
    block_r = 0
    for g in refuse_gaps:
        if g > margin:
            block_r += 1
    print(margin, "| keep answers:", round(100 * keep / len(pos_gaps)), "% | block refuse:", block_r, "/", len(refuse))

0.0 | keep answers: 85 % | block refuse: 10 / 10
0.05 | keep answers: 97 % | block refuse: 10 / 10
0.1 | keep answers: 100 % | block refuse: 8 / 10


In [7]:
MARGIN = 0.05

outside_tests = [
    "ازاي أخس 5 كيلو في شهر؟",
    "ترجملي الجملة دي للألماني",
    "احكيلي نكتة",
    "فين أقرب صيدلية من بيتي في المعادي؟",
    "ما هي عاصمة أستراليا؟",
    "أبي أشتري سيارة جديدة، وش تنصحني؟",
    "My laptop won't turn on, how do I fix it?",
    "What's the best phone to buy this year?",
    "Quel temps fera-t-il à Paris demain ?",
    "Peux-tu m'aider à écrire mon CV ?",
    "Какой курс рубля сегодня?",
    "Расскажи анекдот",
]

hotel_tests = [
    "الواي فاي عندكم ببلاش؟",
    "فين أقرب صيدلية للفندق؟",
    "لو لغيت الحجز قبلها بيومين هيرجعلي فلوسي؟",
    "هل يوجد موقف سيارات في الفندق؟",
    "أبي أعرف متى يقفل السبا؟",
    "Is there a gym I can use at night?",
    "Le petit-déjeuner est-il inclus dans le prix ?",
    "Можно ли поздно выехать из номера?",
]

print("=== outside (should be BLOCK):")
for q in outside_tests:
    g = gap(q)
    if g > MARGIN:
        print("    BLOCK", round(g, 3), q)
    else:
        print("    PASS ", round(g, 3), q)

print("=== hotel (should be PASS):")
for q in hotel_tests:
    g = gap(q)
    if g > MARGIN:
        print("    BLOCK", round(g, 3), q)
    else:
        print("    PASS ", round(g, 3), q)

=== outside (should be BLOCK):
    PASS  -0.003 ازاي أخس 5 كيلو في شهر؟
    BLOCK 0.211 ترجملي الجملة دي للألماني
    BLOCK 0.109 احكيلي نكتة
    PASS  -0.123 فين أقرب صيدلية من بيتي في المعادي؟
    PASS  0.041 ما هي عاصمة أستراليا؟
    PASS  0.041 أبي أشتري سيارة جديدة، وش تنصحني؟
    PASS  0.009 My laptop won't turn on, how do I fix it?
    BLOCK 0.067 What's the best phone to buy this year?
    BLOCK 0.279 Quel temps fera-t-il à Paris demain ?
    BLOCK 0.116 Peux-tu m'aider à écrire mon CV ?
    BLOCK 0.284 Какой курс рубля сегодня?
    BLOCK 0.1 Расскажи анекдот
=== hotel (should be PASS):
    PASS  -0.109 الواي فاي عندكم ببلاش؟
    PASS  -0.092 فين أقرب صيدلية للفندق؟
    PASS  -0.134 لو لغيت الحجز قبلها بيومين هيرجعلي فلوسي؟
    PASS  -0.062 هل يوجد موقف سيارات في الفندق؟
    PASS  -0.129 أبي أعرف متى يقفل السبا؟
    PASS  -0.104 Is there a gym I can use at night?
    PASS  -0.094 Le petit-déjeuner est-il inclus dans le prix ?
    PASS  -0.24 Можно ли поздно выехать из номера?


In [3]:
print(s.search("الجو هيبقى عامل إيه بكرة؟"))
print(len(s.search("الفطار لحد إمتى؟")))

[]
5
